# InferLite on free Google Colab GPUs

This notebook runs **measured** inference experiments (TTFT, tokens/sec, P50/P95/P99, memory, load time) and **labels unsupported methods instead of inventing scores**.

**Setup**

1. Runtime → Change runtime type → **T4 GPU**.
2. File → Upload this notebook, **or** open it from the cloned repo.
3. Run the next cell. It looks for the repo under `/content/llm-inferlite` (GitHub clone) and `/content/llm-inferlite-main` (zip upload). If neither exists, it clones `https://github.com/Shivani767/llm-inferlite`.

Uploading only the `.ipynb` is not enough by itself — InferLite lives in `backend/research/`. The next cell will clone the repo if that folder is missing.

In [ ]:
import os, sys, pathlib, subprocess

GITHUB = "https://github.com/Shivani767/llm-inferlite.git"
CLONE_DEST = pathlib.Path("/content/llm-inferlite")


def is_repo(path: pathlib.Path) -> bool:
    return (path / "backend" / "research").is_dir()


def find_repo():
    names = ("llm-inferlite", "llm-inferlite-main")
    roots = [pathlib.Path.cwd(), pathlib.Path("/content"), pathlib.Path("/content/drive/MyDrive")]
    tried = []
    for root in roots:
        tried.append(root)
        if is_repo(root):
            return root
        for name in names:
            cand = root / name
            tried.append(cand)
            if is_repo(cand):
                return cand
    content = pathlib.Path("/content")
    if content.is_dir():
        for child in sorted(content.iterdir()):
            if child.is_dir() and is_repo(child):
                return child
    return None


REPO_ROOT = find_repo()
if REPO_ROOT is None:
    print(f"Repo not on disk. Cloning {GITHUB} → {CLONE_DEST}")
    if CLONE_DEST.exists() and not is_repo(CLONE_DEST):
        raise RuntimeError(f"{CLONE_DEST} exists but is not an InferLite tree. Delete it and rerun.")
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB, str(CLONE_DEST)])
    REPO_ROOT = CLONE_DEST

BACKEND = REPO_ROOT / "backend"
if not (BACKEND / "research").is_dir():
    raise FileNotFoundError(
        f"backend/research missing under {REPO_ROOT}. "
        "In Colab, either run this cell to auto-clone, or:\n"
        "  !git clone --depth 1 https://github.com/Shivani767/llm-inferlite.git /content/llm-inferlite"
    )

os.chdir(BACKEND)
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print("REPO_ROOT", REPO_ROOT)
print("BACKEND ", BACKEND)
print("cwd     ", pathlib.Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-colab.txt

In [ ]:
from research.capabilities import probe
from research.env import collect_environment
import json

env = collect_environment(seed=42)
caps = probe()
print("device:", caps["device"], "cuda:", caps["cuda"], "colab:", env.get("colab"))
print("gpu:", (env.get("torch") or {}).get("gpu"))
print("\nCapability matrix:")
for name, item in caps["experiments"].items():
    print(f"  [{'YES' if item['supported'] else 'NO ':3}] {name}: {item['reason']}")

## Run the Colab T4 suite

Default model: TinyLlama 1.1B. INT8/INT4 run if bitsandbytes + CUDA work. AWQ, GPTQ, GGUF, TensorRT-LLM, SmoothQuant, SqueezeLLM, and vLLM PagedAttention are attempted and recorded as **unsupported** when libraries or files are missing — they are never given fake TPS.

Speculative decoding uses TinyLlama as both target and draft so tokenizers match. That is a valid measurement of the algorithm, not a claim about a smaller draft model.

In [ ]:
from research.runner import load_config, run_config

cfg_path = REPO_ROOT / "configs" / "colab_t4.yaml"
cfg = load_config(cfg_path)
# Keep the first Colab run short enough for a free session. Increase later.
cfg["max_new_tokens"] = 24
cfg["measure_runs"] = 2
cfg["warmup_runs"] = 1
cfg["results_dir"] = str(BACKEND / "results" / "colab_t4")

summary = run_config(cfg, results_dir=cfg["results_dir"], make_plots=True)
print("measured", summary["n_measured"], "unsupported", summary["n_unsupported"], "error", summary["n_error"])
print("bundle", summary["bundle"])
print("csv", summary["csv"])
print("plots", summary["plots"])
print("pareto", json.dumps(summary["pareto"], indent=2, default=str)[:2000])

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path(cfg["results_dir"]) / "figures"
if fig_dir.exists():
    for p in sorted(fig_dir.glob("*.png")):
        print(p.name)
        display(Image(filename=str(p)))
else:
    print("No figures yet. Check n_measured in the previous cell.")

## Optional: download a GGUF and measure llama.cpp

Skip this cell if `llama-cpp-python` is not installed. InferLite will not fabricate GGUF numbers.

In [ ]:
from research.engine import run_benchmark

try:
    import llama_cpp  # noqa: F401
    rec = run_benchmark(
        model_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        method="gguf",
        backend="llama.cpp",
        gguf_file="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        max_new_tokens=32,
        measure_runs=2,
        warmup_runs=1,
    )
    print(rec.status, rec.reason)
    print(rec.metrics.model_dump() if rec.metrics else None)
except Exception as exc:
    print("GGUF skipped:", type(exc).__name__, exc)

## Honesty checklist

- Do not copy old README tables that listed Llama-3-8B TensorRT-LLM TPS. Those were simulations.
- Cite only `status=measured` rows, with this notebook's environment dump.
- If a method is `unsupported`, report the reason, not a guessed speedup.